### Importing The Required libraries 


In [1]:
import torch
from torch import nn
import torchvision
from transformers import pipeline 
import cv2
from PIL import Image

C:\Users\rajve\miniconda3\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\rajve\miniconda3\Lib\site-packages\keras\src\export\tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):


### Loading the embedding generation model

In [2]:
embedder=pipeline(model="facebook/dinov2-small-imagenet1k-1-layer",
                  task="image-feature-extraction" )

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.
Device set to use cpu


In [3]:
class FinalEmbedder(nn.Module):
    def __init__(self,embedder):
        super().__init__()
        self.embedder=embedder
    def forward(self,images):
        embeddings=[]
        for image in images:
            emb=torch.tensor(self.embedder(image))
            embeddings.append(emb)
        embeddings=torch.stack(embeddings,dim=0)
        final_embedding=torch.mean(embeddings,dim=0)
        return final_embedding

### Testing the embedder 

In [4]:
final_embedder=FinalEmbedder(embedder)

In [5]:
from PIL import Image
image = [
    Image.open(
        r"input\Screenshot 2026-07-27 223924.png"
    ).convert("RGB")
]

In [7]:
print(type(image))
print(image)

<class 'list'>
[<PIL.Image.Image image mode=RGB size=322x334 at 0x1C0FF87D400>]


In [6]:
embeddings=final_embedder(image)

In [8]:
embeddings

tensor([[[ 2.0197,  1.1323,  4.1734,  ...,  1.0991, -0.6842,  2.4292],
         [-3.1881, -3.4502,  4.2963,  ...,  0.8164, -2.6944, -0.5986],
         [-1.4635, -3.9804,  2.1954,  ...,  1.5570,  0.1457, -0.5002],
         ...,
         [ 0.7045, -1.4543,  1.4045,  ...,  3.2305, -0.2538, -3.6278],
         [-1.3619, -2.0464,  2.8759,  ...,  2.7987,  0.6257, -2.5325],
         [-2.5094, -0.6062,  4.0098,  ...,  2.5799, -1.7330, -2.8372]]])